# 02 - Sentiment Analysis
## Amazon Workforce Analytics Project

**Objective:** Apply TextBlob sentiment analysis to measure emotional tone of employee feedback.

**Tools:** TextBlob, Pandas, Matplotlib

**Input:** Cleaned text data  
**Output:** Sentiment scores (polarity, subjectivity) and labels (positive/neutral/negative)

---


## 1. Setup & Install TextBlob

In [ ]:
!pip install textblob -q

from textblob import TextBlob
import pandas as pd
import matplotlib.pyplot as plt

print("✅ Libraries imported!")

## 2. Understanding Sentiment Scores

TextBlob provides two sentiment metrics:

| Metric | Range | Meaning |
|--------|-------|---------|
| **Polarity** | -1 to +1 | Negative ← 0 → Positive |
| **Subjectivity** | 0 to 1 | Factual ← → Opinionated |

**Examples:**
- "I love this job!" → Polarity: +0.8, Subjectivity: 0.9
- "The shift is 10 hours." → Polarity: 0.0, Subjectivity: 0.1
- "Management is terrible." → Polarity: -0.7, Subjectivity: 0.8


## 3. Load Cleaned Data

In [ ]:
# Load cleaned Glassdoor data
glassdoor_df = pd.read_csv('glassdoor_cleaned.csv')
print(f"Glassdoor shape: {glassdoor_df.shape}")

# Load cleaned YouTube data
youtube_df = pd.read_csv('youtube_cleaned.csv')
print(f"YouTube shape: {youtube_df.shape}")

## 4. Calculate Sentiment Scores - Glassdoor

In [ ]:
# Calculate sentiment for PROS
glassdoor_df['pros_polarity'] = glassdoor_df['pros_cleaned'].apply(
    lambda x: TextBlob(str(x)).sentiment.polarity
)
glassdoor_df['pros_subjectivity'] = glassdoor_df['pros_cleaned'].apply(
    lambda x: TextBlob(str(x)).sentiment.subjectivity
)

# Calculate sentiment for CONS
glassdoor_df['cons_polarity'] = glassdoor_df['cons_cleaned'].apply(
    lambda x: TextBlob(str(x)).sentiment.polarity
)
glassdoor_df['cons_subjectivity'] = glassdoor_df['cons_cleaned'].apply(
    lambda x: TextBlob(str(x)).sentiment.subjectivity
)

# Overall sentiment (using combined cleaned text)
glassdoor_df['sentiment_polarity'] = glassdoor_df['cleaned_text'].apply(
    lambda x: TextBlob(str(x)).sentiment.polarity
)
glassdoor_df['sentiment_subjectivity'] = glassdoor_df['cleaned_text'].apply(
    lambda x: TextBlob(str(x)).sentiment.subjectivity
)

print("✅ Glassdoor sentiment calculated!")
glassdoor_df[['pros_polarity', 'cons_polarity', 'sentiment_polarity']].describe()

## 5. Create Sentiment Labels

We categorize polarity scores into labels:
- **Positive:** Polarity > 0.1
- **Negative:** Polarity < -0.1
- **Neutral:** Between -0.1 and 0.1


In [ ]:
def label_sentiment(score):
    """Convert polarity score to sentiment label"""
    if score > 0.1:
        return 'positive'
    elif score < -0.1:
        return 'negative'
    else:
        return 'neutral'

# Apply to Glassdoor
glassdoor_df['sentiment_label'] = glassdoor_df['sentiment_polarity'].apply(label_sentiment)
glassdoor_df['pros_sentiment'] = glassdoor_df['pros_polarity'].apply(label_sentiment)
glassdoor_df['cons_sentiment'] = glassdoor_df['cons_polarity'].apply(label_sentiment)

# Check distribution
print("=== Glassdoor Overall Sentiment ===")
print(glassdoor_df['sentiment_label'].value_counts())
print(f"\nPercentages:")
print(glassdoor_df['sentiment_label'].value_counts(normalize=True).round(2) * 100)

## 6. Calculate Sentiment Scores - YouTube

In [ ]:
# Calculate sentiment for YouTube transcripts
youtube_df['sentiment_polarity'] = youtube_df['transcript_cleaned'].apply(
    lambda x: TextBlob(str(x)).sentiment.polarity
)
youtube_df['sentiment_subjectivity'] = youtube_df['transcript_cleaned'].apply(
    lambda x: TextBlob(str(x)).sentiment.subjectivity
)

# Create sentiment labels
youtube_df['sentiment_label'] = youtube_df['sentiment_polarity'].apply(label_sentiment)

print("=== YouTube Sentiment ===")
print(youtube_df['sentiment_label'].value_counts())
print(f"\nPercentages:")
print(youtube_df['sentiment_label'].value_counts(normalize=True).round(2) * 100)

## 7. Visualize Sentiment Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Glassdoor pie chart
colors = ['#2ecc71', '#95a5a6', '#e74c3c']  # green, gray, red
glassdoor_counts = glassdoor_df['sentiment_label'].value_counts()
axes[0].pie(glassdoor_counts, labels=glassdoor_counts.index, autopct='%1.0f%%', 
            colors=colors, startangle=90)
axes[0].set_title('Glassdoor Sentiment Distribution', fontsize=14, fontweight='bold')

# YouTube pie chart
youtube_counts = youtube_df['sentiment_label'].value_counts()
axes[1].pie(youtube_counts, labels=youtube_counts.index, autopct='%1.0f%%',
            colors=colors, startangle=90)
axes[1].set_title('YouTube Sentiment Distribution', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('sentiment_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("✅ Chart saved: sentiment_comparison.png")

## 8. Cross-Platform Comparison

In [ ]:
# Create comparison table
comparison = pd.DataFrame({
    'Platform': ['Glassdoor', 'YouTube'],
    'Positive %': [
        (glassdoor_df['sentiment_label'] == 'positive').mean() * 100,
        (youtube_df['sentiment_label'] == 'positive').mean() * 100
    ],
    'Neutral %': [
        (glassdoor_df['sentiment_label'] == 'neutral').mean() * 100,
        (youtube_df['sentiment_label'] == 'neutral').mean() * 100
    ],
    'Negative %': [
        (glassdoor_df['sentiment_label'] == 'negative').mean() * 100,
        (youtube_df['sentiment_label'] == 'negative').mean() * 100
    ],
    'Sample Size': [len(glassdoor_df), len(youtube_df)]
})

print("=== Cross-Platform Sentiment Comparison ===")
print(comparison.to_string(index=False))

## 9. Save Results

In [ ]:
# Save Glassdoor with sentiment
glassdoor_df.to_csv('glassdoor_sentiment.csv', index=False)
print("✅ Saved: glassdoor_sentiment.csv")

# Save YouTube with sentiment
youtube_df.to_csv('youtube_sentiment.csv', index=False)
print("✅ Saved: youtube_sentiment.csv")

---
## Key Findings

**Glassdoor vs YouTube Sentiment:**
- Glassdoor reviews show more emotional range (positive AND negative)
- YouTube transcripts are predominantly neutral
- Despite discussing complaints, YouTube shows 0% negative — TextBlob detects emotional *words*, not negative *experiences*

**Insight:** Storytelling tone ≠ sentiment. Platform format shapes how people express opinions.

**Next:** `03_keyword_extraction.ipynb` - Extract keywords and bigrams using NLTK
